## ANN (ARTIFICIAL NEURAL NETWORK)

### Rgression Modelling

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder
import pickle

In [2]:
## Load the dataset
data=pd.read_csv("../Churn_Modelling.csv")


In [3]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


In [5]:
## Preprocess the data
### Drop irrelevant columns
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


Note: Geography and Gender are categorical, which will require some kind of encoding

In [6]:
## Encode categorical variables (gender)
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [7]:
## OneHotEncoder for geography column

ohe=OneHotEncoder()
geo_encoder=ohe.fit_transform(data[['Geography']])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [8]:
encoded_columns=ohe.get_feature_names_out(['Geography'])
encoded_columns

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
## lets convert the encoded columns into pandas dataframe

geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=encoded_columns)

In [10]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [11]:
## combine columns with the original data
data=pd.concat([data.drop('Geography', axis=1),geo_encoded_df],axis=1)

data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [12]:
## Save the encoders and scalar
with open("label_encoder_gender.pkl",'wb') as file:
    pickle.dump(label_encoder_gender,file)

with open("onehot_encoder_geo.pkl", "wb") as file:
    pickle.dump(ohe,file)

In [13]:
## Divide the dataset into dependent and independent
X=data.drop("EstimatedSalary",axis=1)
y=data["EstimatedSalary"]

In [14]:
## Split the data to train and test
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [15]:
## scale the features
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [16]:
with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)

In [17]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [18]:
### Training with regression

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


In [20]:
X_train_scaled.shape

(8000, 12)

In [21]:
X_train_scaled.shape[1] # To get the number of columns

12

In [22]:
## Build the ANN model
model=Sequential( ## sequential connects them all in a sequence
    [
        Dense(64,activation='relu',input_shape=(X_train_scaled.shape[1],)), ## First hidden layer(64 neurons) that is connected to the inputs
        Dense(32,activation='relu'), ## second hidden layer(32 neurons)
        Dense(1) ## output layer ### We wont apply any activation function because the default is 'linear' which is for regression problems
    ]  
)

c:\Users\pc\Downloads\python_deep_learning\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
# import tensorflow
# opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
# loss=tensorflow.keras.losses.MeanAbsoluteError

## We can directly input this but lets just follow KRISH

In [25]:
## Before we do the forward and backward propagation we need to COMPILE the model

model.compile(optimizer="adam",loss="mean_absolute_error",metrics=["mae"]) ## This initialization uses the generic 'adam' and 'binary_crossentropy'. You can initialize them like we did above to make our own adjustments and apply them as arguments

# model.compile(optimizer=opt,loss=loss,metrics=["accuracy"]) 


In [26]:
## set up the tensor board
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir="regressionlogs/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [27]:
## setup early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10, restore_best_weights=True) ## This says monitor the training for validation loss(test set). If there is no improvement Be patient for at least 5 epochs.

In [28]:
## TRAINING THE MODEL
history=model.fit(
    X_train_scaled,y_train,validation_data=(X_test_scaled,y_test),epochs=100,callbacks=[early_stopping_callback,tensorboard_callback,]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 100353.1094 - mae: 100353.1094 - val_loss: 98430.6797 - val_mae: 98430.6797
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 99328.7891 - mae: 99328.7891 - val_loss: 96376.9688 - val_mae: 96376.9688
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 95768.2109 - mae: 95768.2109 - val_loss: 91202.8828 - val_mae: 91202.8828
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 88910.0859 - mae: 88910.0859 - val_loss: 82871.4844 - val_mae: 82871.4844
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 79406.9297 - mae: 79406.9297 - val_loss: 72999.7578 - val_mae: 72999.7578
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 69238.6094 - mae: 69238.6094 - val_loss: 63802.6758 - val_mae: 63802.6758
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 60589.0703 - mae: 60589.0703 - val_loss: 56898.1914 - val_mae: 56898.1914
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step

In [29]:
import os
os.environ["TENSORBOARD_BINARY"] = r"C:/Users/pc/Downloads/python_deep_learning/venv/Scripts/tensorboard.exe" ## We will have to manually assign the path to the TENSORBOARD_BINARY in the environment.

In [30]:
## Load Tensorboard Extension
%load_ext tensorboard

In [31]:
%tensorboard --logdir regressionlogs/fit20261008-155909

Reusing TensorBoard on port 6006 (pid 12196), started 2:19:20 ago. (Use '!kill 12196' to kill it.)

In [32]:
## Evaluate the model on the test data
test_loss,test_mae=model.evaluate(X_test_scaled,y_test)
print(f'Loss: {test_loss}, MAE:{test_mae}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 50426.9453 - mae: 50426.9453
Loss: 50426.9453125, MAE:50426.9453125


In [33]:
## Save the model
model.save("regression_model.keras")

We will carryout deployment later. It follows the same process as we did with ann classification